In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# Улучшенный OZON RecSys Baseline - Рекомендательная система для категории Apparel
# Этот скрипт содержит улучшенное решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.
# Улучшения: Приоритет истории пользователя, включение покупок в prefs, настройка params, валидация.

import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, deque
import heapq
import subprocess
import json
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import gc
import polars as pl
from typing import Iterable, Dict, List, Tuple
from sklearn.metrics import ndcg_score  # Для валидации
import time
import pyarrow as pa
import pyarrow.dataset as ds

pd.options.mode.copy_on_write = True

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# --- компактный буфер последних N уникальных просмотров на пользователя
class RecentUniqueBuffer:
    def __init__(self, maxlen: int = 100):
        self.maxlen = maxlen
        self.heap = []      # (ts_ns:int64, item_id:int)
        self.seen = set()

    def add_raw(self, ts_ns: int, item_id: int):
        if item_id in self.seen:
            return
        heapq.heappush(self.heap, (int(ts_ns), int(item_id)))
        self.seen.add(int(item_id))
        if len(self.heap) > self.maxlen:
            _, old = heapq.heappop(self.heap)
            self.seen.discard(old)

    def to_sorted_list(self):
        return [iid for _, iid in sorted(self.heap, reverse=True)]

## 1. Загрузка и анализ данных с валидационным сплитом

DATA_DIR = "/kaggle/input/testststs"  # Укажите ваш путь
VALIDATION_CUTOFF = pd.to_datetime('2025-06-01')  # Для временной валидации

def load_train_data():

    print("Загружаем тренировочные данные...")
    
    orders_files = sorted(glob.glob(os.path.join(DATA_DIR, "ml_ozon_recsys_train_final_apparel_orders_data", "ml_ozon_recsys_train_final_apparel_orders_data", "*.parquet"), recursive=True))
    tracker_files = sorted(glob.glob(os.path.join(DATA_DIR, "ml_ozon_recsys_train_final_apparel_tracker_data", "ml_ozon_recsys_train_final_apparel_tracker_data", "*.parquet"), recursive=True))
    items_files = sorted(glob.glob(os.path.join(DATA_DIR, "ml_ozon_recsys_train_final_apparel_items_data", "ml_ozon_recsys_train_final_apparel_items_data", "*.parquet"), recursive=True))
    categories_files = sorted(glob.glob(os.path.join(DATA_DIR, "ml_ozon_recsys_train_final_categories_tree", "ml_ozon_recsys_train_final_categories_tree", "*.parquet"), recursive=True))

    print(f"Найдено файлов заказов: {len(orders_files)}")
    print(f"Найдено файлов взаимодействий: {len(tracker_files)}")
    print(f"Найдено файлов товаров: {len(items_files)}")
    print(f"Найдено файлов категорий: {len(categories_files)}")
    
    orders_data = []
    for file_path in tqdm(orders_files, desc="Загрузка заказов"):
        df = pd.read_parquet(file_path, columns=["user_id", "item_id", "last_status", "created_date"])
        orders_data.append(df)
    orders_df = pd.concat(orders_data, ignore_index=True)
    orders_df['created_date'] = pd.to_datetime(orders_df['created_date'])
    print(f"Загружено заказов: {len(orders_df):,}")
    
    # Разделение на train и valid
    train_orders = orders_df[orders_df['created_date'] < VALIDATION_CUTOFF]
    valid_orders = orders_df[orders_df['created_date'] >= VALIDATION_CUTOFF]
    print(f"Train заказов: {len(train_orders):,}, Valid заказов: {len(valid_orders):,}")

    items_data = []
    for file_path in tqdm(items_files, desc="Загрузка товаров"):
        df = pd.read_parquet(file_path, columns=["item_id", "catalogid"])
        items_data.append(df)
    items_df = pd.concat(items_data, ignore_index=True)
    print(f"Загружено товаров: {len(items_df):,}")
    
    return train_orders, valid_orders, items_df

train_orders, valid_orders, items_df = load_train_data()

def human_bytes(n):
    for unit in ["байт","КиБ","МиБ","ГиБ","ТиБ"]:
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} ПиБ"

def mem_df(df):
    return int(df.memory_usage(deep=True).sum())

print("RAM: train_orders =", human_bytes(mem_df(train_orders)))
print("RAM: valid_orders =", human_bytes(mem_df(valid_orders)))
print("RAM: items_df  =", human_bytes(mem_df(items_df)))

print("order_moment min/max:", train_orders["created_date"].min(), train_orders["created_date"].max())

## 2. Exploratory Data Analysis (EDA)

print("АНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(train_orders):,}")
print(f"Уникальных пользователей: {train_orders['user_id'].nunique():,}")
print(f"Уникальных товаров: {train_orders['item_id'].nunique():,}")
print(f"Период данных: {train_orders['created_date'].min()} - {train_orders['created_date'].max()}")

print("\nРаспределение статусов заказов:")
status_counts = train_orders['last_status'].value_counts()
for status, count in status_counts.items():
    percentage = count / len(train_orders) * 100
    print(f"  {status}: {count:,} ({percentage:.1f}%)")

print("\nАНАЛИЗ ТОВАРОВ")
print("=" * 50)
print(f"Общее количество товаров: {len(items_df):,}")

print("\nТоп-10 категорий:")
top_categories = items_df['catalogid'].value_counts().head(10)
for cat_id, count in top_categories.items():
    print(f"  Категория {cat_id}: {count:,}")

## 3. Загрузка тестовых пользователей

def load_test_users():

    print("Загружаем тестовых пользователей...")
    
    test_files = sorted(glob.glob(os.path.join(DATA_DIR, "ml_ozon_recsys_test.snappy.parquet"), recursive=True))
    all_users = set()
    
    for file_path in tqdm(test_files, desc="Обработка тестовых файлов"):
        df = pd.read_parquet(file_path)
        all_users.update(df['user_id'].unique())
    
    print(f"Найдено уникальных тестовых пользователей: {len(all_users):,}")
    return list(all_users)

test_users = load_test_users()

## 4. Построение модели на основе популярности

# --- Функции весов (как раньше)
def _freshness_weight(days_ago: pd.Series, alpha: float) -> pd.Series:
    return 1.0 / (1.0 + alpha * days_ago.clip(lower=0))

def _seasonality_weight(event_dt: pd.Series, reference_date: pd.Timestamp, beta: float) -> pd.Series:
    m_now = reference_date.month
    m_evt = event_dt.dt.month
    dist = (m_now - m_evt).abs()
    dist = dist.where(dist <= 6, 12 - dist)
    w = 1.0 + beta * (1.0 - dist / 6.0)
    return w.clip(0.5, 1.5)

def compute_reference_date_safe(orders_df, order_time_col="created_date", dataset_end="2025-07-01"):
    od = pd.to_datetime(orders_df.get(order_time_col), errors="coerce")
    ref = od.max()
    if pd.isna(ref):
        ref = pd.Timestamp(dataset_end)
    ds_end = pd.Timestamp(dataset_end)
    if ref > ds_end:
        ref = ds_end
    return pd.to_datetime(ref)

## === STREAM: популярность + пользовательские предпочтения (в одном проходе по трекеру) ===

TRACKER_DIR = os.path.join(DATA_DIR, "ml_ozon_recsys_train_final_apparel_tracker_data", "ml_ozon_recsys_train_final_apparel_tracker_data")

def build_popularity_and_userprefs_streaming(
    orders_df: pd.DataFrame,
    items_df: pd.DataFrame,
    test_users: Iterable[int],
    *,
    tracker_dir: str = TRACKER_DIR,
    # веса/гиперпараметры
    alpha: float = 0.03, beta: float = 0.20,
    w_buy: float = 5.0, w_view: float = 1.0,
    # имена колонок
    user_col: str = "user_id",
    item_col: str = "item_id",
    order_time_col: str = "created_date",
    order_status_col: str = "last_status",
    track_time_col: str = "date",
    track_action_col: str = "action_type",
    view_label: str = "page_view",
    items_item_col: str = "item_id",
    items_catalog_col: str = "catalogid",
    delivered_labels=("delivered", "delivered_orders", "доставлен"),
    # управление скоростью/памятью
    pyarrow_batch_size: int = 5_000_000,   # «подсказка» сканеру Arrow (реально ограничено row-group)
    target_chunk_rows: int = 15_000_000,  # «супербатч»: склеиваем несколько RB перед pandas
    log_every_batches: int = 20,         # как часто печатать прогресс
    # выход
    top_k: int = 500,
    cap_per_category: int = 10,
    cap_penalty: float = 0.05,
    # предпочтения пользователей
    use_purchases_in_popularity=True,
    use_purchases_in_prefs=True,
    max_views_per_user: int = 100,
    max_purchases_per_user: int = 100,
    reference_date: pd.Timestamp | None = None,
) -> Tuple[List[int], Dict[int, List[int]]]:
    """
    Возвращает:
      - popular_items (list[int]) — топ-K popular с кэпом по catalogid
      - user_preferences (dict[user_id -> list[item_id]]) — последние покупки (опц.) + просмотры

    Всё считается за ОДИН проход по parquet-трекеру.
    """
    print("▶️ STREAM: популярность + пользовательские предпочтения (одним проходом)")
    t0 = time.time()
    print(
        f"flags: use_purchases_in_popularity={use_purchases_in_popularity}, "
        f"use_purchases_in_prefs={use_purchases_in_prefs}"
    )

    # --- reference_date
    if reference_date is None:
        reference_date = compute_reference_date_safe(orders_df, order_time_col)
    reference_date = pd.to_datetime(reference_date)

    m_now = reference_date.month
    lut = np.array(
        [1.0 + beta * (1.0 - min(abs(m_now - m), 12 - abs(m_now - m)) / 6.0) for m in range(1, 13)],
        dtype="float64",
    )
    lut = np.clip(lut, 0.5, 1.5)

    print("reference_date:", reference_date.date())

    # --- 0) Покупки (обычно помещаются в RAM)
    combined = defaultdict(float)          # item_id -> score (популярность)
    if use_purchases_in_popularity:
        if order_status_col in orders_df.columns:
            st = orders_df[order_status_col].astype(str).str.lower()
            delivered_mask = False
            for lab in delivered_labels:
                delivered_mask |= st.str.contains(str(lab).lower())
            delivered = orders_df.loc[delivered_mask, [item_col, order_time_col]].copy()
        else:
            delivered = orders_df[[item_col, order_time_col]].copy()

        if not delivered.empty:
            delivered[order_time_col] = pd.to_datetime(delivered[order_time_col], errors="coerce")
            days = (
                reference_date - delivered[order_time_col]
            ).dt.days.fillna(0).clip(lower=0).astype(int)
            w = (
                w_buy
                * _freshness_weight(days, alpha)
                * _seasonality_weight(delivered[order_time_col], reference_date, beta)
            )
            buy_scores = w.groupby(delivered[item_col]).sum()
            for iid, sc in buy_scores.items():
                combined[int(iid)] += float(sc)
            print(f"  ✓ покупки: учтено {len(buy_scores):,} item_id")
            print(
                f"  ✓ покупки→популярность: учтено {len(buy_scores):,} item_id, "
                f"строк доставленных: {len(delivered):,}"
            )
            del delivered, days, w, buy_scores
            gc.collect()
        else:
            print("  ⚠️ покупки: таблица пустая, пропускаем")
            print("  ⚠️ покупки→популярность: доставленных нет/не распарсились — пропуск")

    # --- 1) Подготовка к проходу по трекеру
    test_users = pd.Index(test_users).astype(int)
    users_set = set(test_users.tolist())
    # буферы для последних просмотров по пользователям
    user_buffers: Dict[int, RecentUniqueBuffer] = {}

    dataset = ds.dataset(tracker_dir, format="parquet")
    # ВАЖНО: для популярности берём все page_view; для предпочтений отберём test_users внутри батча.
    filt = (ds.field(track_action_col) == view_label)
    cols = [item_col, track_time_col, track_action_col, user_col]  # добавили user_id, чтобы собирать prefs

    # scanner кросс-версийно
    try:
        scanner = dataset.scanner(columns=cols, filter=filt, batch_size=pyarrow_batch_size).use_threads(True)
    except AttributeError:
        scanner = ds.Scanner.from_dataset(dataset, columns=cols, filter=filt, batch_size=pyarrow_batch_size)

    reader = scanner.to_reader()

    processed_rows = 0
    total_batches = 0
    chunk_rows = 0
    chunk_batches = []

    def _process_chunk(batches):
        nonlocal processed_rows
        if not batches:
            return

        tbl = pa.Table.from_batches(batches)
        pdf = tbl.to_pandas()
        n = len(pdf)
        processed_rows += n
        if n == 0:
            del pdf, tbl
            gc.collect()
            return

        # 1) подготовка
        if track_action_col in pdf.columns:
            pdf = pdf.drop(columns=[track_action_col])

        # отбрасываем NaT в time
        pdf[track_time_col] = pd.to_datetime(pdf[track_time_col], errors="coerce")
        m = pdf[track_time_col].notna()
        if not m.any():
            del pdf, tbl
            gc.collect()
            return

        # только нужные 3 колонки текущего супербатча
        sub = pdf.loc[m, [item_col, track_time_col, user_col]]

        # ===== Популярность по просмотрам (векторно, без groupby) =====
        # ts -> дни
        ts = sub[track_time_col].values.astype("datetime64[ns]").astype("int64")      # int64 ns
        ref_ns = np.int64(reference_date.value)
        days = (ref_ns - ts) // np.int64(86_400_000_000)
        days = np.maximum(days, 0, dtype=np.int64)

        # веса свежести + сезонности через LUT (ты её уже посчитал выше)
        w_fresh = 1.0 / (1.0 + alpha * days)                                         # float64
        months = sub[track_time_col].dt.month.to_numpy()         # 1..12
        w_seas = lut[months - 1]
        vals = w_view * w_fresh * w_seas                                             # веса на строках

        # аккумулируем item-очки через factorize+bincount (быстрее groupby)
        keys = sub[item_col].to_numpy(copy=False)
        codes, uniques = pd.factorize(keys, sort=False)
        acc = np.bincount(codes, weights=vals)
        for iid, sc in zip(uniques, acc):
            combined[int(iid)] += float(sc)

        # ===== Пользовательские просмотры для test_users =====
        # векторный отбор нужных юзеров
        user_ids = sub[user_col].to_numpy(copy=False)
        item_ids = sub[item_col].to_numpy(copy=False)
        sel = np.isin(user_ids, test_users.values)   # test_users — твой pd.Index(int)

        if sel.any():
            ts_ns = ts  # уже есть int64 наносекундные отметки
            for u, iid, tns in zip(user_ids[sel], item_ids[sel], ts_ns[sel]):
                buf = user_buffers.get(int(u))
                if buf is None:
                    buf = user_buffers[int(u)] = RecentUniqueBuffer(max_views_per_user)
                buf.add_raw(tns, int(iid))

        del pdf, tbl, sub
        gc.collect()

    t_scan = time.time()
    for rb in reader:  # rb: pyarrow.RecordBatch
        total_batches += 1
        chunk_batches.append(rb)
        chunk_rows += rb.num_rows

        if total_batches % log_every_batches == 0:
            print(
                f"[scan] batch #{total_batches}: rb_rows={rb.num_rows:,}, "
                f"chunk_rows={chunk_rows:,}, processed_rows~{processed_rows:,}, "
                f"users_with_views~{len(user_buffers):,}"
            )

        if chunk_rows >= target_chunk_rows:
            _process_chunk(chunk_batches)
            chunk_batches.clear()
            chunk_rows = 0

    _process_chunk(chunk_batches)
    t_end_scan = time.time()
    print(
        f"✓ Трекер обработан: {processed_rows:,} строк за {t_end_scan - t_scan:.1f} сек; "
        f"пользователей с просмотрами: {len(user_buffers):,}"
    )

    # --- 2) Базовый рейтинг
    if not combined:
        print("⚠️ Нет данных для популярности.")
        popular_items = []
    else:
        base = pd.DataFrame(list(combined.items()), columns=[item_col, "score"])
        base.sort_values(["score", item_col], ascending=[False, True], inplace=True, kind="mergesort")
        print("Уникальных товаров в рейтинге:", f"{len(base):,}")

        # кэп по catalogid ≤ cap_per_category
        map_df = (
            items_df[[items_item_col, items_catalog_col]]
            .drop_duplicates(subset=[items_item_col])
            .rename(columns={items_item_col: item_col, items_catalog_col: "catalogid"})
        )
        ranked = base.merge(map_df, on=item_col, how="left")
        miss = ranked["catalogid"].isna().sum()
        if miss:
            ranked["catalogid"] = ranked["catalogid"].fillna("__unknown__")
            print(f"ℹ️ catalogid не найден для {miss:,} товаров — __unknown__")

        ranked["rank_in_cat"] = ranked.groupby("catalogid").cumcount() + 1
        mask_cap = ranked["rank_in_cat"] > cap_per_category
        n_pen = int(mask_cap.sum())
        if n_pen:
            ranked.loc[mask_cap, "score"] = ranked.loc[mask_cap, "score"] * cap_penalty
        ranked.sort_values(["score", item_col], ascending=[False, True], inplace=True, kind="mergesort")
        popular_items = ranked[item_col].head(top_k).tolist()
        print(f"Кэп применён: penalized {n_pen:,}, итоговый top-{top_k} готов.")

    # --- 3) Пользовательские предпочтения
    # Покупки (опционально) — берём последние max_purchases_per_user на пользователя теста
    user_preferences: Dict[int, List[int]] = {}
    test_users_list = test_users.tolist()
    purchased: Dict[int, List[int]] = {}
    if use_purchases_in_prefs:
        df = orders_df[[user_col, item_col, order_time_col, order_status_col]].copy()
        st = df[order_status_col].astype(str).str.lower()
        delivered_mask = False
        for lab in delivered_labels:
            delivered_mask |= st.str.contains(str(lab).lower())
        df = df.loc[delivered_mask]
        if not df.empty:
            df = df[df[user_col].isin(test_users_list)]
            df[order_time_col] = pd.to_datetime(df[order_time_col], errors="coerce")
            df = df.dropna(subset=[order_time_col])
            df.sort_values([user_col, order_time_col], ascending=[True, False], inplace=True)
            df["rn"] = df.groupby(user_col).cumcount()
            df = df[df["rn"] < max_purchases_per_user]
            purchased = df.groupby(user_col)[item_col].apply(list).to_dict()
        print(f"[prefs] пользователей с покупками: {len(purchased):,}")
    else:
        print("[prefs] ⏭ покупки в предпочтения не добавляем (флаг выключен)")

    # формируем финальный список: покупки (если надо) + просмотры (из буферов), без дублей
    cnt_hist = 0
    for u in test_users_list:
        acc = []
        seen = set()
        if use_purchases_in_prefs and u in purchased:
            for iid in purchased[u]:
                if iid not in seen:
                    acc.append(iid)
                    seen.add(iid)
        if u in user_buffers:
            for iid in user_buffers[u].to_sorted_list():
                if iid not in seen:
                    acc.append(iid)
                    seen.add(iid)
        user_preferences[u] = acc
        if acc:
            cnt_hist += 1

    print(f"✓ Предпочтения готовы: {cnt_hist:,}/{len(test_users_list):,} пользователей с историей.")
    print(f"⏱️ Общее время: {time.time() - t0:.1f} сек.")
    return popular_items, user_preferences

# 1. Опорная дата (reference_date) на основе заказов
safe_ref = compute_reference_date_safe(
    train_orders,
    order_time_col="created_date",
    dataset_end="2025-07-01"
)

# 2. Запуск стриминговой функции
popular_items, user_preferences = build_popularity_and_userprefs_streaming(
    train_orders,
    items_df,
    test_users,
    tracker_dir=TRACKER_DIR,
    alpha=0.03,
    beta=0.20,
    w_buy=5.0,
    w_view=1.0,
    top_k=500,
    cap_per_category=10,
    cap_penalty=0.05,
    max_views_per_user=100,
    max_purchases_per_user=100,
    reference_date=safe_ref,
    use_purchases_in_popularity=True,
    use_purchases_in_prefs=True
)

# 3. Быстрый sanity-check
print("Top-10 популярных товаров:", popular_items[:10])

any_user = next(iter(user_preferences))
print(f"Пример предпочтений для пользователя {any_user}:", user_preferences[any_user][:10])

## 6. Генерация рекомендаций

def generate_recommendations(test_users, popular_items, user_preferences):

    print("Генерируем рекомендации...")
    
    recommendations = {}
    
    for user_id in tqdm(test_users, desc="Создание рекомендаций"):
        user_hist = set(user_preferences.get(user_id, []))
        
        user_recs = list(user_hist)[:100]  # Приоритет истории
        
        if len(user_recs) < 100:
            remaining = 100 - len(user_recs)
            fillers = [item for item in popular_items if item not in user_hist][:remaining]
            user_recs.extend(fillers)
            
            if len(user_recs) < 100:
                remaining = 100 - len(user_recs)
                random_items = np.random.choice(popular_items, remaining, replace=True)
                user_recs.extend(random_items)
        
        recommendations[user_id] = user_recs[:100]
    
    print(f"Сгенерированы рекомендации для {len(recommendations):,} пользователей")
    
    rec_lengths = [len(recs) for recs in recommendations.values()]
    print(f"Средняя длина рекомендаций: {np.mean(rec_lengths):.1f}")
    print(f"Все рекомендации имеют 100 товаров: {all(l == 100 for l in rec_lengths)}")
    
    return recommendations

recommendations = generate_recommendations(test_users, popular_items, user_preferences)

## Валидация на holdout

def validate_recs(valid_orders, recommendations):
    valid_purchases = valid_orders[valid_orders['last_status'].str.contains('delivered', case=False)]
    user_actual = valid_purchases.groupby('user_id')['item_id'].apply(list).to_dict()
    scores = []
    for user in user_actual:
        if user in recommendations:
            pred = recommendations[user]
            actual = [1 if item in user_actual[user] else 0 for item in pred]
            scores.append(ndcg_score([actual], [np.arange(len(pred), 0, -1)], k=100))
    if scores:
        print(f"Валидационный NDCG@100: {np.mean(scores):.4f}")
    else:
        print("Нет данных для валидации.")

validate_recs(valid_orders, recommendations)

## 7. Создание файла submission

def create_submission(recommendations, filename='improved_baseline_submission.csv'):

    print(f"Создаем файл submission: {filename}")
    
    submission_data = []
    for user_id, items in tqdm(recommendations.items(), desc="Формирование submission"):
        submission_data.append({
            'user_id': user_id,
            'item_id_1 item_id_2 ... item_id_100': ' '.join(map(str, items))
        })
    
    submission_df = pd.DataFrame(submission_data)
    submission_df.to_csv(filename, index=False)
    
    print(f"Сохранен submission с {len(submission_df):,} пользователями")
    print(f"Размер файла: {Path(filename).stat().st_size / 1024 / 1024:.1f} MB")
    
    return filename

submission_file = create_submission(recommendations)

Загружаем тренировочные данные...
Найдено файлов заказов: 5
Найдено файлов взаимодействий: 200
Найдено файлов товаров: 100
Найдено файлов категорий: 1


Загрузка заказов: 100%|██████████| 5/5 [00:03<00:00,  1.29it/s]


Загружено заказов: 18,887,122
Train заказов: 15,050,203, Valid заказов: 3,836,919


Загрузка товаров: 100%|██████████| 100/100 [00:03<00:00, 27.19it/s]


Загружено товаров: 6,439,422
RAM: train_orders = 1.4 ГиБ
RAM: valid_orders = 353.5 МиБ
RAM: items_df  = 73.7 МиБ
order_moment min/max: 2025-01-01 00:00:00 2025-05-31 00:00:00
АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 15,050,203
Уникальных пользователей: 808,119
Уникальных товаров: 4,064,250
Период данных: 2025-01-01 00:00:00 - 2025-05-31 00:00:00

Распределение статусов заказов:
  delivered_orders: 8,421,207 (56.0%)
  canceled_orders: 6,508,533 (43.2%)
  proccesed_orders: 120,463 (0.8%)

АНАЛИЗ ТОВАРОВ
Общее количество товаров: 6,439,422

Топ-10 категорий:
  Категория 7559: 413,637
  Категория 7508: 267,444
  Категория 36484: 225,336
  Категория 7660: 219,858
  Категория 7502: 199,919
  Категория 7512: 120,846
  Категория 7511: 108,535
  Категория 7530: 106,773
  Категория 17002: 105,158
  Категория 7644: 101,622
Загружаем тестовых пользователей...


...

▶️ STREAM: популярность + пользовательские предпочтения (одним проходом)
flags: use_purchases_in_popularity=True, use_purchases_in_prefs=True
reference_date: 2025-05-31
  ✓ покупки: учтено 2,752,891 item_id
  ✓ покупки→популярность: учтено 2,752,891 item_id, строк доставленных: 8,421,207
[scan] batch #20: rb_rows=2,235,600, chunk_rows=8,096,360, processed_rows~50,344,882, users_with_views~465,302
[scan] batch #40: rb_rows=2,337,329, chunk_rows=14,050,017, processed_rows~103,090,464, users_with_views~468,472
[scan] batch #60: rb_rows=2,636,667, chunk_rows=2,636,667, processed_rows~174,233,057, users_with_views~469,467
[scan] batch #80: rb_rows=2,248,091, chunk_rows=8,149,238, processed_rows~227,598,281, users_with_views~469,746
[scan] batch #100: rb_rows=2,189,458, chunk_rows=13,948,138, processed_rows~280,323,257, users_with_views~469,884
[scan] batch #120: rb_rows=2,294,383, chunk_rows=2,294,383, processed_rows~351,310,445, users_with_views~470,011
[scan] batch #140: rb_rows=2,392,923

Создание рекомендаций: 100%|██████████| 470347/470347 [00:35<00:00, 13194.68it/s]


Сгенерированы рекомендации для 470,347 пользователей
Средняя длина рекомендаций: 100.0
Все рекомендации имеют 100 товаров: True
Валидационный NDCG@100: 0.1799
Создаем файл submission: improved_baseline_submission.csv


Формирование submission: 100%|██████████| 470347/470347 [00:20<00:00, 22539.81it/s]


Сохранен submission с 470,347 пользователями
Размер файла: 438.7 MB


In [4]:
xxx = pd.read_csv("/kaggle/working/improved_baseline_submission.csv")
print(xxx.head(100))

    user_id                item_id_1 item_id_2 ... item_id_100
0         1  237398019 97589767 264852487 284625935 4561870...
1   3145730  150821510 251103750 159984905 266331155 243243...
2   3145731  18210309 134201368 243747356 227357226 9677060...
3   1048580  327411712 15327251 235491347 43389972 28895081...
4   4194310  140044289 338132495 317058589 14990371 1245235...
..      ...                                                ...
95  1048771  199146503 248969740 119461395 40725524 3344693...
96  4194501  199660033 55847430 102361618 183551001 1404170...
97      200  83657218 272866822 29306887 8320014 330575890 ...
98  3145930  148573701 129233942 166383639 306398762 329803...
99  1048781  203548161 32549894 274836492 149136915 2545741...

[100 rows x 2 columns]
